##### Copyright 2020 HrFlow's AI Research Department

Licensed under the Apache License, Version 2.0 (the "License");

In [ ]:
# Copyright 2020 HrFlow's AI Research Department. All Rights Reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# ==============================================================================

# Overview
This notebook is made to evaluate the scoring of profiles regarding a specific job or jobs regarding specific profile
This notebook will be structured as follow:
* General functions:
    * Get all items
    * Get scoring results
    * Tag item
* Score profiles for a specific job
* Score jobs for a specific profile

In [2]:
import requests
import pandas as pd
import json
import os
from datetime import datetime
from tqdm import tqdm
from hrflow import Hrflow
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
API_SECRET = os.getenv("API_SECRET")
API_USER = os.getenv("API_USER")
ALGORITHM = os.getenv("ALGORITHM")
BOARD_KEY = os.getenv("BOARD_KEY")
BOARD_KEYS = os.getenv("BOARD_KEYS")
SOURCE_KEY = os.getenv("SOURCE_KEY")
SOURCE_KEYS = os.getenv("SOURCE_KEYS")
OUTPUT_FILE = os.getenv("OUTPUT_FILE")
LIMIT_SCORING = "32"
LIMIT_SEARCHING = "10000"
ALGORITHM_FAMILY = "tagger-rome4-family"
ALGORITHM_SUBFAMILY = "tagger-rome4-subfamily"
ALGORITHM_CATEGORY = "tagger-rome4-category"
ALGORITHM_JOB_TITLE = "tagger-rome4-jobtitle"

In [ ]:
SOURCE_KEYS=json.loads(SOURCE_KEYS)
BOARD_KEYS=json.loads(BOARD_KEYS)

In [ ]:
client = Hrflow(api_secret=API_SECRET,api_user=API_USER)

In [ ]:
## Function to score item based on source or board of items
def get_scoring_items(client, item, source_keys=None, board_keys=None):
    if source_keys:
        response_scoring = client.profile.scoring.list(
            job_key=item["key"],
            board_key=BOARD_KEY,
            source_keys=source_keys,
            limit=LIMIT_SCORING,
            agent_key=ALGORITHM
        )
    else:
        assert board_keys is not None
        response_scoring = client.job.scoring.list(
            profile_key=item["key"],
            source_key=SOURCE_KEY,
            board_keys=board_keys,
            limit=LIMIT_SCORING,
            agent_key=ALGORITHM
        )
    
    if response_scoring["code"] != 200:
        print("error while returning scoring:", response_scoring)
        return
    
    scored_items = response_scoring["data"]["profiles"] if source_keys else response_scoring["data"]["jobs"]

    scores = [prediction[1] for prediction in response_scoring["data"]["predictions"]]
    
    return item, scored_items, scores

# get items is sources or boards
def get_items_searching(
    client,source_keys=None,board_keys=None
):
    if source_keys:
        response_searching = client.profile.searching.list(
            source_keys=source_keys,
            limit=LIMIT_SEARCHING,
            order_by="desc"
        )
    else:
        assert board_keys is not None
        response_searching = client.job.searching.list(
            board_keys=board_keys,
            limit=LIMIT_SEARCHING,
            order_by="desc"
        )
    
    if response_searching["code"] != 200:
        print("error while returning searching:", response_searching)
        return
    
    searched_items = response_searching["data"]["profiles"] if source_keys else response_searching["data"]["jobs"]
    return searched_items

## function to tag items
def tagger_romev4(text, algorithm):
    url = "https://api.hrflow.ai/v1/text/tagging"

    payload = {
        "algorithm_key": algorithm,
        "texts": [text],
        "top_n": 1,
    }
    headers = {
        "accept": "application/json",
        "content-type": "application/json",
        "X-API-KEY": API_SECRET,
        "X-USER-EMAIL": API_USER,
    }

    response = requests.post(url, json=payload, headers=headers)
    if response.status_code != 200:
        print(f"HTTP error: {response.text}")
        return None

    response_data = response.json()
    
    data = response_data.get("data")
    if data and isinstance(data[0], dict):
        tags = data[0].get("tags")
        if tags and isinstance(tags, list):
            return tags[0] if tags else None
        
    return None

def format_date(date_str: str) -> str:
    if date_str:
        return datetime.strptime(date_str, "%Y-%m-%dT%H:%M:%S+0000").strftime("%Y-%m-%d")
    return None

def categorize_scores(scores):
    star_count = {
        "5 stars": sum(0.8 <= score <= 1 for score in scores),
        "4 stars": sum(0.6 <= score < 0.8 for score in scores),
        "3 stars": sum(0.4 <= score < 0.6 for score in scores),
        "2 stars": sum(0.2 <= score < 0.4 for score in scores),
        "1 star": sum(0 <= score < 0.2 for score in scores),
    }
    return star_count


## Score profiles for a specific job

In [ ]:
jobs = get_items_searching(client,board_keys=BOARD_KEYS)
for job in tqdm(jobs):
    sections_desc = "\n".join([section["description"] for section in job["sections"]])
    
    tags_data = {
        "family": tagger_romev4(sections_desc, ALGORITHM_FAMILY),
        "subfamily": tagger_romev4(sections_desc, ALGORITHM_SUBFAMILY),
        "category": tagger_romev4(sections_desc, ALGORITHM_CATEGORY),
        "job_title": tagger_romev4(sections_desc, ALGORITHM_JOB_TITLE),
    }

    job["tags"].extend(
        [{"name": f"hrflow_tag_romev4_{key}", "value": value} for key, value in tags_data.items() if value]
    )
        

In [ ]:
scoring_job_result = []
for job in tqdm(jobs):
    scoring_result = get_scoring_items(client,job,source_keys=SOURCE_KEYS)
    scoring_job_result.append(scoring_result)

rows = []
for job, _, scores in scoring_job_result:
    star_count = categorize_scores(scores)
    tags = job["tags"]

    sections_desc = "\n".join([section["description"] for section in job["sections"]])
    len_offres = len(sections_desc)
    
    family = None
    subfamily = None
    category = None
    job_title = None
        
    for tag in tags:
        if tag["name"] == "hrflow_tag_romev4_family":
            family = tag["value"]
        if tag["name"] == "hrflow_tag_romev4_subfamily":
            subfamily = tag["value"]
        if tag["name"] == "hrflow_tag_romev4_category":
            category = tag["value"]
        if tag["name"] == "hrflow_tag_romev4_job_title":
            job_title = tag["value"]
    
    rows.append({
        "Nom": job["name"],
        "Reference": job["reference"],
        "Date de création" : format_date(job["created_at"]),
        "Localisation" : job["location"]["text"],
        "Nombre de caractères de l'offre": len_offres ,
        "Nombre de profils ayant 5 étoiles": star_count["5 stars"],
        "Nombre de profils ayant 4 étoiles": star_count["4 stars"],
        "Nombre de profils ayant 3 étoiles": star_count["3 stars"],
        "Nombre de profils ayant 2 étoiles": star_count["2 stars"],
        "Nombre de profils ayant 1 étoiles": star_count["1 star"],
        "Tagger romev4 family": family,
        "Tagger romev4 subfamily": subfamily,
        "Tagger romev4 category": category,
        "Tagger romev4 job title":  job_title,
    })

df = pd.DataFrame(rows)


df.to_excel(OUTPUT_FILE, index=False)

print(f"Excel file '{OUTPUT_FILE}' generated successfully.")

## Score jobs for a specific profile

In [ ]:
profiles = get_items_searching(client,source_keys=SOURCE_KEYS)
for profile in tqdm(profiles):
    tags_data = {
        "family": tagger_romev4(profile["text"], ALGORITHM_FAMILY),
        "subfamily": tagger_romev4(profile["text"], ALGORITHM_SUBFAMILY),
        "category": tagger_romev4(profile["text"], ALGORITHM_CATEGORY),
        "job_title": tagger_romev4(profile["text"], ALGORITHM_JOB_TITLE),
    }

    profile["tags"].extend(
        [{"name": f"hrflow_tag_romev4_{key}", "value": value} for key, value in tags_data.items() if value]
    )
        

In [ ]:
scoring_profile_result = []
for profile in tqdm(profiles):
    scoring_result = get_scoring_items(client,profile,board_keys=BOARD_KEYS)
    scoring_profile_result.append(scoring_result)

rows = []
for profile, _, scores in scoring_profile_result:
    star_count = categorize_scores(scores)
    tags = profile["tags"]

    family = None
    subfamily = None
    category = None
    job_title = None
        
    for tag in tags:
        if tag["name"] == "hrflow_tag_romev4_family":
            family = tag["value"]
        if tag["name"] == "hrflow_tag_romev4_subfamily":
            subfamily = tag["value"]
        if tag["name"] == "hrflow_tag_romev4_category":
            category = tag["value"]
        if tag["name"] == "hrflow_tag_romev4_job_title":
            job_title = tag["value"]
    
    rows.append({
        "Nom": profile["name"],
        "Reference": profile["reference"],
        "Date de reception" : format_date(profile["created_at"]),
        "Localisation" : profile["location"]["text"],
        "Nombre de caractères de l'offre": len_offres ,
        "Nombre de jobs ayant 5 étoiles": star_count["5 stars"],
        "Nombre de jobs ayant 4 étoiles": star_count["4 stars"],
        "Nombre de jobs ayant 3 étoiles": star_count["3 stars"],
        "Nombre de jobs ayant 2 étoiles": star_count["2 stars"],
        "Nombre de jobs ayant 1 étoiles": star_count["1 star"],
        "Tagger romev4 family": family,
        "Tagger romev4 subfamily": subfamily,
        "Tagger romev4 category": category,
        "Tagger romev4 job title":  job_title,
    })

df = pd.DataFrame(rows)


df.to_excel(OUTPUT_FILE, index=False)

print(f"Excel file '{OUTPUT_FILE}' generated successfully.")